![Platform+](images/banner.png)

# Lab 7: Getting Data In
## Stage → Format → COPY INTO → Query → Transform → Automate

> **Persona track: Data Engineer** — same lab, your angle.
>
> Getting data in is only step one. An engineer owns the *pipeline*, not just the load — it has to survive bad files and run without you. In this lab you'll follow one dataset through the engineer's loop: **land it → load it → handle bad rows → transform semi-structured JSON → automate the reload with a task.**

:point_right: **In this lesson, you'll bring two datasets into Snowflake: loyalty sign-ups from an internal stage, and a supplier product catalog from a dedicated supplier stage. You'll define file formats, load both files with COPY INTO, handle bad data with ON_ERROR strategies, query nested JSON with colon-dot path notation, and schedule a load with a lightweight task.**

The scenario: The Roastery loyalty programme has been running for months — sign-up data lives in an internal stage. Meanwhile, your coffee bean supplier drops quarterly product catalog updates to a separate supplier stage. You need to get both into Snowflake reliably — and eventually automate the reload cycle.

We'll cover:
1. Inspecting what's in a stage before loading
2. Defining a CSV file format and loading with COPY INTO
3. Connecting to the supplier stage and loading the product catalog
4. Catching bad data with VALIDATION_MODE, then loading with ON_ERROR=CONTINUE
5. Your angle — querying JSON with colon-dot path notation and scheduling COPY INTO with a task

:books: **By the end you'll be able to:**
- Use `LIST` and `SELECT $1, $2, $3` to inspect a staged file before loading
- Create a FILE FORMAT and use it with COPY INTO
- Compare VALIDATION_MODE (dry-run) vs ON_ERROR=CONTINUE (partial load)
- Query semi-structured JSON from a VARIANT column using colon-dot path notation
- Create a scheduled task that automates COPY INTO

**Time:** ~50 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set your session context

Every query runs under a role (what you're allowed to do) and a warehouse (compute). The cells below create your personal warehouse and database if they don't exist yet, then switch into your own schema for this lab.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RAW;
USE SCHEMA {{my_db}}.RAW;


---
## 1 · Inspect Before You Load

Good loading practice: always look at the raw file before writing a single line of SQL. You need to know the column count, delimiter, whether there's a header row, and any obvious data issues.

The stage `PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE` holds the loyalty sign-up data. Let's see what's there.

In [ ]:
LIST @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE;


> :mag_right: **Review the output.** You should see `loyalty_signups.csv`. Note the file size — you'll verify the row count after loading.

**Reading from a stage without loading.** The `SELECT` below doesn't move any data into Snowflake — it streams the raw file from the stage and returns each row as text. Nothing is written anywhere.

The `$1`, `$2`, `$3` notation is **positional**: Snowflake maps each delimited field in the file to a numbered column — `$1` is the first field, `$2` the second, and so on. Because no **FILE FORMAT** has been applied yet, every field comes back as a raw string. Adding `$4` shows the *boundary*: an empty `$4` means field 4 doesn't exist, confirming exactly 3 columns. Use this to answer the three questions you need: *How many columns are there? Does the file have a header row? What's the delimiter?*

> 💡 **What's a FILE FORMAT?** A FILE FORMAT is the small set of instructions that tells Snowflake how to interpret the shape of a file: its delimiter, whether it has a header row, how nulls are written. You'll create one in the next step.

In [ ]:
-- Positional preview: $1/$2/$3 = the three fields, $4 empty = no 4th column (boundary check)
SELECT $1, $2, $3, $4
FROM @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE/loyalty_signups.csv
LIMIT 5;


**See the raw line.** Turn off delimiter parsing with `FIELD_DELIMITER='NONE'` so `$1` returns the whole line — now you can see the real delimiter and any quoting yourself.

In [ ]:
CREATE OR REPLACE FILE FORMAT {{my_db}}.RAW.NO_DELIM_FMT TYPE='CSV' FIELD_DELIMITER='NONE';
SELECT $1 AS raw_line
FROM @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE/loyalty_signups.csv
  (FILE_FORMAT => '{{my_db}}.RAW.NO_DELIM_FMT')
LIMIT 5;


> :mag_right: **Your turn — read the file.**  
> Before building the FILE FORMAT you need to answer three questions about this file:
> 1. **How many columns are there?** (Look at which positional columns have values — and which is empty.)
> 2. **Does the file have a header row?** (Look at row 1: is it data or labels?)
> 3. **What's the delimiter?** (The filename is a clue — what does `.csv` stand for?)
>
> <details><summary>Check your read</summary>
>
> **Three columns** — `$1`, `$2`, and `$3` all have values; `$4` is empty in every row, marking the column boundary.  
> **Row 1 is the header** — it shows `CUSTOMER_ID`, `SIGNUP_DATE`, `CHANNEL` rather than actual values. You'll need to skip it with `SKIP_HEADER=1`.  
> **Comma-delimited CSV** — `.csv` stands for comma-separated values, so the delimiter is a comma. `CSV` is exactly the value you'll enter for `TYPE` in the FILE FORMAT cell coming up next. The date format is `YYYY-MM-DD` and channel is a short string (`app`, `web`, `in-store`).
>
> That's all the information you need to write the FILE FORMAT.
> </details>

---
## 2 · Define the Format, Then COPY INTO

Snowflake uses **FILE FORMAT** objects to describe how a staged file is structured. You define it once and reuse it across COPY INTO statements.

**If you don't define a file format, Snowflake assumes CSV** — comma-delimited fields and newline-delimited rows. That default is fine for a simple comma-separated file, but the moment your file has a header row, a different delimiter, or a different type (like JSON), you have to spell it out.

**Why this matters for engineers:** A wrong FILE FORMAT — missing `SKIP_HEADER`, wrong delimiter, incorrect encoding — is the most common cause of pipeline load failures. Understanding the format object means you can diagnose any load error in under a minute.

> :warning: **The cell below is wrong on purpose.** It declares the format as `TYPE = JSON` with `SKIP_HEADER = 0`. Run it anyway — Snowflake will reject it. Read the error: it tells you an option isn't valid for JSON, which is your clue that this isn't a JSON file at all. It's a CSV with a header row. **Fix the two settings** to what you learned from the peek:
> - `TYPE` — the *kind* of file you're loading. The file extension and peek confirmed it's a comma-separated-values file, so this is `CSV` (not JSON).
> - `SKIP_HEADER` — how many rows at the top are labels rather than data. The peek showed a single header row, so this is `1` (not 0).
>
> The rest is already correct for you: `FIELD_DELIMITER=','` (fields split on commas), `TRIM_SPACE=TRUE` (strip stray spaces), `NULL_IF=('','NULL')` (treat blanks and the literal text `NULL` as null), and `EMPTY_FIELD_AS_NULL=TRUE`. Reference solution is just below the cell.

In [ ]:
-- WRONG ON PURPOSE: TYPE should not be JSON, and SKIP_HEADER should not be 0.
-- Run it, read the error, then fix TYPE and SKIP_HEADER (reference is just below the cell).
CREATE OR REPLACE FILE FORMAT {{my_db}}.RAW.CSV_FMT
  TYPE           = JSON
  SKIP_HEADER    = 0
  FIELD_DELIMITER = ','
  TRIM_SPACE     = TRUE
  NULL_IF        = ('', 'NULL')
  EMPTY_FIELD_AS_NULL = TRUE;


> <details><summary>📖 Reference solution — stuck on the two values?</summary>
>
>     TYPE        = CSV   -- comma-separated-values file (not JSON)
>     SKIP_HEADER = 1     -- skip the single header row (not 0)
>
> - **TYPE = CSV** — tells Snowflake to parse the file as delimited text.
> - **SKIP_HEADER = 1** — ignores row 1 (the column names) so it isn't loaded as a data row.
> - **FIELD_DELIMITER = ','** — fields are separated by commas.
> - **TRIM_SPACE / NULL_IF / EMPTY_FIELD_AS_NULL** — strip stray whitespace and turn blanks or the literal text `NULL` into real NULLs.
> </details>

### Now create the table to load into

The FILE FORMAT describes *how to read* the file. Next you need somewhere to put the data: a table whose columns line up with the file's fields. `COPY INTO` maps them **by position** — field 1 → column 1, field 2 → column 2, and so on — so the column order below must match the order you saw in the peek (`customer_id`, `signup_date`, `channel`), and each column needs a type that fits its values.

The types are already filled in for you, but you should understand why: `customer_id` is `INT` because the peek showed numeric IDs; `signup_date` is `DATE` so you can do date math on it later; `channel` is `VARCHAR(30)` because it holds short text values.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.LOYALTY_SIGNUPS (
  customer_id  INT,
  signup_date  DATE,
  channel      VARCHAR(30)
);


This is your first **COPY INTO** — the command that actually loads staged data into a table. It has a *lot* of optional parameters (error handling, file selection, transformations, validation). You don't need to memorise them; you need to know they exist and how to look them up.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">What are the main parameters of the Snowflake COPY INTO &lt;table&gt; command? Group them by purpose (file location, file format, error handling, load behaviour) and give a one-line description of each.</pre>

> :point_right: **Your turn.** The COPY INTO below is missing its file format name. Fill in the name of the format you just fixed (`FORMAT_NAME = ______`), then run it. Reference solution is just below the cell.

In [ ]:
COPY INTO {{my_db}}.RAW.LOYALTY_SIGNUPS
FROM @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE/loyalty_signups.csv
FILE_FORMAT = (FORMAT_NAME = ______);  -- fill in your file format name


> <details><summary>📖 Reference solution</summary>
>
>     FILE_FORMAT = (FORMAT_NAME = {{my_db}}.RAW.CSV_FMT)
> </details>

In [ ]:
SELECT COUNT(*) AS rows_loaded FROM {{my_db}}.RAW.LOYALTY_SIGNUPS;


> :mag_right: **Review the output.** The COPY INTO result shows rows loaded, parse errors (should be 0), and file status (LOADED). Confirm COUNT(*) matches.

> :point_right: **Your turn.** Run a `SELECT * FROM {{my_db}}.RAW.LOYALTY_SIGNUPS LIMIT 10` to confirm column types are correct — `customer_id` should be INT, `signup_date` a DATE, `channel` a VARCHAR.

In [ ]:
SELECT * FROM {{my_db}}.RAW.LOYALTY_SIGNUPS LIMIT 10;


---
## 3 · Connect to the Supplier Stage

The loyalty sign-ups came from an internal stage — data your team controls. But many real-world sources come from *outside*: a vendor, a partner, a cloud-based system writing files to its own bucket.

**The scenario:** Roastery's coffee bean suppliers send quarterly product catalog updates to a dedicated supplier stage. The latest Q4 2025 catalog just arrived. Let's load it.

Start by listing the stage to see what the supplier has dropped.

In [ ]:
LIST @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE;


> :mag_right: **Review the output.** Four files: `roastery_products_q4_2025.csv` (clean catalog — load this now), `roastery_products_q4_2025_errors.csv` (bad rows — used in Section 5), `roastery_loyalty_events.json` (semi-structured events — Section 5), and `roastery_products_q4_2025_quoted.csv` (a quoted-field variant used in the Platform Admin lab — ignore here).

In [ ]:
SELECT $1, $2, $3, $4, $5, $6
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025.csv
LIMIT 4;


> :mag_right: **Review the output.** Row 1 is the header: `product_id`, `product_name`, `category`, `origin_country`, `cost_per_unit`, `active`. The data is clean — numeric IDs, decimal costs, TRUE/FALSE flags.

---

**You just listed an internal stage.** In production, a supplier wouldn't write to your internal stage — they'd write to their own cloud storage bucket, and you'd point an **external stage** at it. You don't set that up here (that's a platform-admin job), but it's worth understanding what changes.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Conceptually, what's the difference between an internal stage and an external stage in Snowflake? If a supplier dropped these files in their own S3/Azure/GCS bucket instead, what would a Snowflake external stage need in order to read from it (URL, credentials/authorization)? Keep it high-level — I just want to understand what changes, not set up cloud IAM.</pre>

> 📌 **Note:** This lab uses an internal stage so you can run it without cloud infrastructure. When the **infrastructure administrator** provisions the supplier's S3 bucket, only the `CREATE STAGE` definition changes — the COPY INTO syntax you write below stays exactly the same.

**Same three steps as before.** Loading the supplier catalog follows the exact pattern you just used for the sign-ups: **(1)** define a file format that matches the file, **(2)** create the target table, **(3)** run COPY INTO. You've seen it once — now you'll build more of it yourself.

> 💡 **Why a second file format?** You already have `CSV_FMT` — but here you'll author a format from scratch for the practice. In real pipelines each source often gets its own named format, so this mirrors production.

> :point_right: **Your turn.** The file format below is only started for you — just the `CREATE` line and `TYPE = CSV`. This is a CSV with a header row, so add the option it needs to skip that header. (Everything else you might add is already the default, so `SKIP_HEADER` is the one line that matters here.) Reference solution is just below the cell.

In [ ]:
CREATE OR REPLACE FILE FORMAT {{my_db}}.RAW.CSV_SUPPLIER_FMT
  TYPE = CSV
  -- your turn: add the option this file needs (it has a header row)


> <details><summary>📖 Reference solution</summary>
>
>     SKIP_HEADER = 1
> </details>

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.SUPPLIER_PRODUCTS (
  product_id      INT,
  product_name    VARCHAR(100),
  category        VARCHAR(50),
  origin_country  VARCHAR(50),
  cost_per_unit   FLOAT,
  active          BOOLEAN
);


> :point_right: **Your turn — write the whole statement.** You've filled in a `FORMAT_NAME` and completed a file format. Now write the entire COPY INTO from scratch in the blank cell below: load `roastery_products_q4_2025.csv` from `@PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE` into `SUPPLIER_PRODUCTS`, using the `CSV_SUPPLIER_FMT` format you just built. Then run it. Reference solution is just below the cell.

In [ ]:
-- Your turn: write the COPY INTO here, then run.


> <details><summary>📖 Reference solution</summary>
>
>     COPY INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS
>     FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025.csv
>     FILE_FORMAT = (FORMAT_NAME = {{my_db}}.RAW.CSV_SUPPLIER_FMT);
> </details>

> :warning: **Getting "Numeric value 'product_id' is not recognized"?** That means the header row is being loaded as data — your `CSV_SUPPLIER_FMT` is missing `SKIP_HEADER = 1`. Fix the format above, then re-run the `COPY INTO` cell. Paste the error into CoCo if you want it to walk you through the fix.

In [ ]:
-- Spot-check the data itself (the COPY INTO output above already reported the row count)
SELECT * FROM {{my_db}}.RAW.SUPPLIER_PRODUCTS LIMIT 5;


> :mag_right: **Review the output.** The COPY INTO result above already confirmed the load — check its **`rows_parsed`** and **`rows_loaded`** (50 each, 0 errors). This cell spot-checks the actual rows so you can eyeball types and values before moving on.

#### Reshape while you load 🔧

You just loaded the catalog *as-is* — every column, same order, same types. But COPY INTO can also **transform on the way in**: give it a target column list and a `FROM (SELECT …)` over the stage, and you can **cast** types, **reorder or drop** columns, and **add lineage** columns — no separate staging table required.

Here you'll load the same file into a leaner table: cast `cost_per_unit` to `NUMBER(10,2)` (money, not a float), **drop** `origin_country`, and **add** a `source_file` column that records which file each row came from using the built-in `METADATA$FILENAME`.

> :point_right: **Your turn.** One expression is blank — add the metadata column that captures the source filename (`METADATA$FILENAME`). Then run it. Reference solution is just below the cell.

> :warning: **What a transformation COPY *can't* do.** The `SELECT` here is deliberately limited to a **simple projection** — casts, reordering, dropping columns, and `METADATA$` columns. It does **not** support `WHERE`, `ORDER BY`, `GROUP BY`, or `LIMIT` (Snowflake rejects it: *"COPY statement only supports simple SELECT from stage"*). To filter or aggregate, load first, then transform in SQL.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.SUPPLIER_PRODUCTS_SHAPED (
  product_id     INT,
  product_name   VARCHAR(100),
  category       VARCHAR(50),
  cost_per_unit  NUMBER(10,2),
  source_file    VARCHAR
);


In [ ]:
-- Transform on load: cast cost, drop origin_country, add a lineage column.
COPY INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS_SHAPED
     (product_id, product_name, category, cost_per_unit, source_file)
FROM (
  SELECT $1::INT,
         $2::STRING,
         $3::STRING,
         $5::NUMBER(10,2),
         ______                 -- add the lineage column: the file each row came from
  FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025.csv
       (FILE_FORMAT => '{{my_db}}.RAW.CSV_SUPPLIER_FMT')
);


> <details><summary>📖 Reference solution</summary>
>
>     METADATA$FILENAME
>
> `METADATA$FILENAME` is a pseudo-column Snowflake exposes on any staged file — it returns the path of the file each row was read from, which is invaluable for auditing multi-file loads.
> </details>

In [ ]:
-- Note: cost_per_unit is now NUMBER(10,2), origin_country is gone, source_file is populated
SELECT product_id, product_name, cost_per_unit, source_file
FROM {{my_db}}.RAW.SUPPLIER_PRODUCTS_SHAPED
ORDER BY product_id
LIMIT 5;


> :mag_right: **Review the output.** Same 50 rows, but reshaped at load: `cost_per_unit` is a typed decimal, `origin_country` is gone, and every row carries its `source_file`. In a real pipeline this is how you land data in the shape downstream tables expect — without an intermediate copy.

---
## 4 · Handle Load Errors

The supplier also sent a file with two rows where `cost_per_unit` contains text (`N/A`, `PENDING`) instead of a number. Here's the full error-handling toolkit:

| Strategy | What it does | When to use |
|---|---|---|
| `VALIDATION_MODE=RETURN_ALL_ERRORS` | Dry-run: finds all bad rows, writes nothing | When you need a complete error report before loading |
| `ON_ERROR=ABORT_STATEMENT` | **Default.** Fails the whole load on the first bad row; nothing loads | Trusted/controlled feeds where any error means something is genuinely wrong — fail fast |
| `ON_ERROR=SKIP_FILE` | Skips the entire file if it has *any* error | Multi-file loads where one bad file shouldn't taint the batch |
| `ON_ERROR=SKIP_FILE_<n>` / `SKIP_FILE_<n>%` | Skips a file once errors exceed a count or percentage | Tolerate a little noise, but reject a mostly-broken file (e.g. `SKIP_FILE_5%`) |
| `ON_ERROR=CONTINUE` | Loads clean rows, skips bad ones | When partial data is better than no data and you'll backfill |

> :bulb: **How to choose.** Default to fail-fast (`ABORT_STATEMENT`) for sources you control. Dry-run with `VALIDATION_MODE` when onboarding an unknown feed. Use `SKIP_FILE_n%` for tolerant multi-file loads. Reach for `CONTINUE` **only** when partial data beats no data — and always pair it with a capture step (you'll use `VALIDATE` and `COPY_HISTORY` next). The golden rule: a partial load is only safe if the rejected rows are captured and reprocessed.

**Why this matters for engineers:** Choosing the wrong strategy means either blocking a pipeline on a fixable error (`ABORT_STATEMENT`) or silently loading incomplete data with no error report (`CONTINUE` without monitoring). Know both tools.

In [ ]:
-- Strategy 1: dry-run — see all errors without writing anything
COPY INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025_errors.csv
FILE_FORMAT  = (FORMAT_NAME = {{my_db}}.RAW.CSV_SUPPLIER_FMT)
VALIDATION_MODE = RETURN_ALL_ERRORS;


> :mag_right: **Review the output.** Two rows flagged. The `ERROR` column shows which column failed and what value was received. The `ROW_NUMBER` tells you exactly where in the file the problem is.

> ⚙️ **Your angle starts here.** Everything above was shared foundation — staging, formats, COPY INTO. From here on, you apply it as an engineer: handling errors gracefully, parsing semi-structured data, and automating the reload so it runs without you.

---
## 5 · Your angle — Data Engineer: errors, JSON, and a scheduled task

### Part A · Load with ON_ERROR=CONTINUE

Now load the error file with `ON_ERROR=CONTINUE` — Snowflake loads the clean rows and skips the bad ones. Then check exactly how many rows were skipped.

**Engineering note:** When you use `ON_ERROR=CONTINUE`, always check `COPY_HISTORY` or re-query the row count afterward. Silent skips are the most common cause of "where did my data go?" incidents in data pipelines.

In [ ]:
-- Load the "errors" file (52 data rows: 50 clean + 2 with a bad cost_per_unit)
-- into a FRESH table. ON_ERROR=CONTINUE loads the 50 good rows and skips the 2 bad ones.
CREATE OR REPLACE TABLE {{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL (
  product_id      INT,
  product_name    VARCHAR(100),
  category        VARCHAR(50),
  origin_country  VARCHAR(50),
  cost_per_unit   FLOAT,
  active          BOOLEAN
);

COPY INTO {{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025_errors.csv
FILE_FORMAT  = (FORMAT_NAME = {{my_db}}.RAW.CSV_SUPPLIER_FMT)
ON_ERROR     = CONTINUE;


**Why check this yourself?** The COPY INTO result *did* show `rows_parsed` and `rows_loaded` — but that message is transient and session-scoped; it's gone once you move on. Deriving the source total straight from the file and comparing it to what landed gives you a **reusable, auditable assertion** you can bake into a pipeline — one that still works next quarter when the supplier sends a bigger file, without hard-coding any number.

In [ ]:
-- Get the total straight from the source file, then compare to what loaded
WITH src AS (
  SELECT COUNT(*) AS rows_in_source_file
  FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_products_q4_2025_errors.csv
       (FILE_FORMAT => '{{my_db}}.RAW.CSV_SUPPLIER_FMT')
),
loaded AS (
  SELECT COUNT(*) AS rows_loaded FROM {{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL
)
SELECT
    rows_in_source_file,
    rows_loaded,
    rows_in_source_file - rows_loaded AS rows_skipped
FROM src, loaded;


> :mag_right: **Review the output.** `rows_in_source_file` = 52, `rows_loaded` = 50, `rows_skipped` = 2. Sourcing the total from the file itself (rather than hard-coding it) means this check still works if the supplier sends a bigger file next quarter.

**Which rows did we lose?** Skipping bad rows is only safe if you can *see* what was skipped and send it back for reprocessing. The `VALIDATE` table function replays the errors from your most recent COPY (`JOB_ID => '_last'`) and hands you the exact rejected records.

In [ ]:
SELECT ROW_NUMBER, COLUMN_NAME, ERROR, REJECTED_RECORD
FROM TABLE(VALIDATE({{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL, JOB_ID => '_last'));


> :mag_right: **Review the output.** Two rows. `REJECTED_RECORD` shows the raw line from the file — the `N/A` and `PENDING` values in `cost_per_unit`. `ERROR` explains why each failed. **This is exactly what you'd send back to the supplier** (or route to a quarantine table) to fix and reload.

**Confirm it in the audit trail.** Every committed COPY INTO is recorded in `COPY_HISTORY`. This is the same view you'd query weeks later to answer "did that load actually run, and did it have errors?"

In [ ]:
SELECT FILE_NAME, ROW_COUNT, ROW_PARSED, ERROR_COUNT, STATUS, LAST_LOAD_TIME
FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
    TABLE_NAME => '{{my_db}}.RAW.SUPPLIER_PRODUCTS_PARTIAL',
    START_TIME => DATEADD('hour', -1, CURRENT_TIMESTAMP())))
ORDER BY LAST_LOAD_TIME DESC;


> :mag_right: **Review the output.** `ROW_COUNT` = 50 (loaded), `ROW_PARSED` = 52 (read), `ERROR_COUNT` = 2, `STATUS` = `Partially loaded`. The COPY result message is transient; `COPY_HISTORY` is the permanent record.

**Wrapping up error handling.** You now have the full toolkit: **`VALIDATION_MODE`** to see errors before committing, **`ON_ERROR`** (`ABORT_STATEMENT` / `CONTINUE` / `SKIP_FILE[_n%]`) to decide what happens when rows are bad, **`VALIDATE`** to recover the exact rejected records, and **`COPY_HISTORY`** as the permanent audit trail. In a real pipeline you'd combine them: dry-run a new feed, load with a deliberate strategy, and monitor every run. Next, a different shape of data entirely — semi-structured JSON.

---
### Part B · Query semi-structured JSON with colon-dot path notation

The supplier stage also contains `roastery_loyalty_events.json` — a JSON array of loyalty events (signups, purchases, reviews), each with a nested `metadata` object. You'll load it into a **[VARIANT](https://docs.snowflake.com/en/sql-reference/data-types-semistructured) column**, then pull typed columns out of it with **colon-dot path notation**.

**What is semi-structured data?** It's data that carries its own structure — keys, nesting, arrays — instead of fitting neatly into fixed rows and columns. JSON, Avro, ORC, and Parquet are all semi-structured. Snowflake stores it natively in a **VARIANT** column: no schema-on-write, no up-front table design. You load the whole document, then reach into it at query time.

A single VARIANT value holds the whole nested object, and you address into it by path:

    JSON document (one VARIANT value in column "v")     colon-dot path                    result
    ──────────────────────────────────────────────     ────────────────────────          ──────────
    { "event_id":    "EVT-002",                        →   v:event_id::STRING            →  'EVT-002'
      "customer_id": 10087,                            →   v:customer_id::INT            →  10087
      "event_type":  "purchase",                       →   v:event_type::STRING          →  'purchase'
      "metadata": {
          "product_id": 101,                           →   v:metadata.product_id::INT    →  101
          "score":      4,                             →   v:metadata.score::INT         →  4
          "channel":   "web" } }                       →   v:metadata.channel::STRING    →  'web'

The **colon** (`:`) steps from the column into a top-level key; a further **colon** (or a dot) steps into a nested key; the **`::TYPE`** suffix casts the extracted value to a real SQL type. This **colon-dot path notation** is Snowflake's dedicated syntax for traversing semi-structured data — a `:` steps into a key, a further `:` or `.` steps into a nested key, and `::TYPE` casts the result. Full reference: [Querying Semi-structured Data](https://docs.snowflake.com/en/user-guide/querying-semistructured).

**Why this matters for engineers:** semi-structured feeds — event streams, API payloads, logs — are everywhere in modern pipelines. Landing them as-is in a VARIANT and querying them relationally, without a rigid schema, is a core engineering skill.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.LOYALTY_EVENTS_RAW (v VARIANT);


> :point_right: **Your turn.** The COPY INTO below is missing its format **TYPE**. This file isn't CSV — look at the extension. Fill in the correct `TYPE` for a `.json` file before running (hint: it's the extension, uppercased). The `STRIP_OUTER_ARRAY = TRUE` option tells Snowflake the file is one big array and each element should become its own row.

In [ ]:
COPY INTO {{my_db}}.RAW.LOYALTY_EVENTS_RAW
FROM @PLATPLUS_DB.ROASTERY_RAW.SUPPLIER_STAGE/roastery_loyalty_events.json
FILE_FORMAT = (TYPE = ______ STRIP_OUTER_ARRAY = TRUE);  -- fill in the TYPE


In [ ]:
-- Each row is one JSON event stored as a VARIANT
SELECT v FROM {{my_db}}.RAW.LOYALTY_EVENTS_RAW LIMIT 3;


> :mag_right: **Review the output.** Each row is a complete JSON object. You can see the nested structure: `event_id`, `customer_id`, `event_type`, `event_ts`, and a `metadata` object containing `product_id`, `score`, and `channel`.

In [ ]:
-- Extract individual fields from the VARIANT using colon (:) dot-path notation
SELECT
    v:event_id::STRING      AS event_id,
    v:customer_id::INT      AS customer_id,
    v:event_type::STRING    AS event_type,
    v:event_ts::TIMESTAMP   AS event_ts,
    v:metadata.product_id::INT  AS product_id,
    v:metadata.score::INT       AS score,
    v:metadata.channel::STRING  AS channel
FROM {{my_db}}.RAW.LOYALTY_EVENTS_RAW
ORDER BY event_ts
LIMIT 10;


> :mag_right: **Review the output.** You should see 10 rows with clean typed columns extracted from the JSON. Notice that rows where `score` or `product_id` is `null` in the JSON come through as NULL in SQL — exactly what you'd want.

**You have a choice with semi-structured data.** Snowflake lets you **leave it as-is** in a VARIANT and query it with path syntax (schema-on-read — great while the shape is still changing or you're exploring), **or transform/flatten** it into typed columns when you want performance, governance, and BI-tool-friendly output. Flattening comes in two flavours: a **view** flattens *logically* (nothing is copied — the extraction runs at query time), while a **table** flattens *physically* (the typed columns are materialised). Next you'll build the view flavour.

**Your turn — flatten once, query many times.** The cell below does two things in one shot: it creates a view, `LOYALTY_EVENTS_VW`, that flattens the JSON into typed columns using the colon-dot pattern, then queries that view to summarise events by type and channel. Two extraction expressions **in the view** are left blank — fill them in:
- **event_type** lives at the *top level* → `v:event_type::STRING`.
- **channel** lives *inside `metadata`* → `v:metadata.channel::STRING`.

Once the view exists, notice the summary query underneath reads plain column names — no colon-dot. That's the point of the view: the messy extraction lives in one place, and everyone downstream queries clean columns.

In [ ]:
-- Flatten the JSON once into a reusable view, then query it — both in one cell
CREATE OR REPLACE VIEW {{my_db}}.RAW.LOYALTY_EVENTS_VW AS
SELECT
    v:event_id::STRING          AS event_id,
    v:customer_id::INT          AS customer_id,
    ______                      AS event_type,   -- top-level event_type, cast to STRING
    v:event_ts::TIMESTAMP       AS event_ts,
    v:metadata.product_id::INT  AS product_id,
    v:metadata.score::INT       AS score,
    ______                      AS channel       -- channel from inside metadata, cast to STRING
FROM {{my_db}}.RAW.LOYALTY_EVENTS_RAW;

SELECT event_type, channel, COUNT(*) AS event_count
FROM {{my_db}}.RAW.LOYALTY_EVENTS_VW
GROUP BY event_type, channel
ORDER BY event_count DESC;


> <details><summary>📖 Reference solution</summary>
>
>     v:event_type::STRING        AS event_type,
>     ...
>     v:metadata.channel::STRING  AS channel
> </details>

---
### Part C · Schedule COPY INTO with a task

Right now you run COPY INTO manually. In a real pipeline you'd want it to run automatically on a schedule. Snowflake **tasks** let you schedule any single SQL statement.

Every task has the same anatomy (this is pseudo-code — don't run it):

    CREATE OR REPLACE TASK <task_name>
      WAREHOUSE = <warehouse>      -- compute the task runs on
      SCHEDULE  = '<n> MINUTE'     -- how often (interval) — or USING CRON <expr> for calendar times
    AS
      <any single SQL statement>;  -- what runs on each scheduled run

- **`WAREHOUSE`** — the compute it uses (omit it entirely for a serverless task).
- **`SCHEDULE`** — use an interval like `'15 MINUTE'` for "every N minutes"; use `USING CRON ...` only when you need a specific time of day or day of week.
- **`AS <statement>`** — the work itself. For us, a COPY INTO.

A newly created task is **suspended** by default — you start it with `ALTER TASK ... RESUME` and stop it with `ALTER TASK ... SUSPEND`.

> :bulb: **This is just a taster.** Tasks are Snowflake's **built-in engine for schedule-based (and event-driven) execution** — no external scheduler like cron or Airflow **is required** to run SQL on a cadence. There's a lot more to them than we'll cover here: chaining tasks into graphs (**Directed Acyclic Graphs, or DAGs**), triggering on new data with streams, serverless compute, error notifications, and CRON schedules. For now we're just proving the basic loop works.

**Now build the real one — with CoCo.** This is your first time using CoCo's **`/snowflake-tasks`** skill — a purpose-built helper for authoring, scheduling, and troubleshooting Snowflake tasks. Instead of memorising the `CREATE TASK` DDL, you describe what you want and it generates correct, current syntax. You'll still read and run the SQL yourself, so you learn the shape of it.

Instead of writing the DDL by hand, hand CoCo the COPY INTO you want to schedule and let its `/snowflake-tasks` skill generate the task. Notice the prompt asks for **two** statements: the `CREATE TASK`, and an `ALTER TASK ... SUSPEND` right after it so the task can't run and burn credits during the lab.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/snowflake-tasks — create a task named LOAD_SIGNUPS_TASK on warehouse {{my_wh}} that runs every 15 minutes and executes this statement:

COPY INTO {{my_db}}.RAW.LOYALTY_SIGNUPS FROM @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE/loyalty_signups.csv FILE_FORMAT = (FORMAT_NAME = {{my_db}}.RAW.CSV_FMT) ON_ERROR = CONTINUE;

Give me two statements: (1) the CREATE OR REPLACE TASK, and (2) immediately after it, an ALTER TASK that SUSPENDs it so it does not run or consume credits. Return just the SQL.</pre>

> :point_right: **Copy CoCo's SQL into the blank cell below and run it.** If your database/warehouse come through as placeholders, swap in the real names from the setup cell output at the top of the notebook. Reference solution is just below the cell.

> :bulb: **Why suspend it if new tasks start suspended anyway?** They do — a freshly created task won't run until you `RESUME` it. The extra `ALTER TASK … SUSPEND` is belt-and-braces: it guarantees the task is parked no matter what CoCo generates (for example, if it also added a `RESUME`), so nothing runs on a schedule during the lab.

In [ ]:
-- Paste CoCo's two statements on the blank line below (CREATE TASK, then ALTER TASK ... SUSPEND), then run.


> <details><summary>📖 Reference solution</summary>
>
>     CREATE OR REPLACE TASK {{my_db}}.RAW.LOAD_SIGNUPS_TASK
>       WAREHOUSE = {{my_wh}}
>       SCHEDULE  = '15 MINUTE'
>     AS
>       COPY INTO {{my_db}}.RAW.LOYALTY_SIGNUPS
>       FROM @PLATPLUS_DB.ROASTERY_RAW.LOAD_STAGE/loyalty_signups.csv
>       FILE_FORMAT = (FORMAT_NAME = {{my_db}}.RAW.CSV_FMT)
>       ON_ERROR = CONTINUE;
>
>     ALTER TASK {{my_db}}.RAW.LOAD_SIGNUPS_TASK SUSPEND;
> </details>

In [ ]:
SHOW TASKS IN SCHEMA {{my_db}}.RAW;


> :mag_right: **Review the output.** Look at the `state` column — it should show `suspended` (exactly what you want in a lab: the task exists but won't run). The `schedule` column confirms the 15-minute interval, and `definition` shows the SQL the task will run.

---
### Part D · See it run — trigger the task on demand

Your task is scheduled but suspended, so it won't fire on its own. You don't have to wait 15 minutes to prove it works: **[`EXECUTE TASK`](https://docs.snowflake.com/en/sql-reference/sql/execute-task)** triggers a single run right now — the exact same run the schedule would have kicked off.

Let's prove the pipeline end-to-end: empty the table, trigger the task, and watch the rows come back.

In [ ]:
-- Empty the table so we can watch the task refill it
TRUNCATE TABLE {{my_db}}.RAW.LOYALTY_SIGNUPS;
SELECT COUNT(*) AS rows_after_truncate FROM {{my_db}}.RAW.LOYALTY_SIGNUPS;


> :mag_right: **Review the output.** `rows_after_truncate` = 0 — the table is empty. (Truncating also clears the file-load history, so the task's COPY INTO treats the file as new and will load it again.)

In [ ]:
-- Manually fire a single run of the task — the same run the 15-minute schedule would trigger
EXECUTE TASK {{my_db}}.RAW.LOAD_SIGNUPS_TASK;


> :mag_right: **Review the output.** You'll see *"Task ... is scheduled to run immediately."* `EXECUTE TASK` is **asynchronous** — it queues the run and returns straight away; the COPY INTO runs in the background a moment later. (Triggering a run does **not** un-suspend the task — it stays suspended after this one run, so no credits are burned on a schedule.)

**First, confirm the run finished.** `EXECUTE TASK` is asynchronous, so check its status before you look for rows. This reads the task's run history — wait for `STATE = SUCCEEDED`.

In [ ]:
SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, ERROR_MESSAGE
FROM TABLE({{my_db}}.INFORMATION_SCHEMA.TASK_HISTORY(TASK_NAME => 'LOAD_SIGNUPS_TASK'))
ORDER BY SCHEDULED_TIME DESC
LIMIT 5;


> :mag_right: **Review the output.** The most recent row should reach `STATE = SUCCEEDED` within a few seconds (it may briefly show `EXECUTING`). If it shows `FAILED`, `ERROR_MESSAGE` tells you why. Once it's `SUCCEEDED`, poll for the rows below.

**Now poll for the data.** Run the cell below. If it still shows `0`, the background run hasn't finished yet — just run it again every few seconds. Within a few seconds the rows reappear — the same count you loaded back in Section 2 (75 in this file) — now loaded by the task instead of by hand.

In [ ]:
SELECT COUNT(*) AS rows_loaded FROM {{my_db}}.RAW.LOYALTY_SIGNUPS;


> :mag_right: **Review the output.** Once `rows_loaded` matches your Section 2 count (75 for this file), the task has run successfully — you've watched a scheduled pipeline execute on demand, then reload the table from the stage.
>
> :bulb: You can also check any run's status (and error messages) in `{{my_db}}.INFORMATION_SCHEMA.TASK_HISTORY` — filter with `TASK_NAME => 'LOAD_SIGNUPS_TASK'`.

---
## Verify Your Work

In [ ]:
checks = [
    ('LOYALTY_SIGNUPS loaded', f'SELECT COUNT(*) FROM {my_db}.RAW.LOYALTY_SIGNUPS', 1),
    ('SUPPLIER_PRODUCTS loaded', f'SELECT COUNT(*) FROM {my_db}.RAW.SUPPLIER_PRODUCTS', 1),
    ('SUPPLIER_PRODUCTS_SHAPED loaded', f'SELECT COUNT(*) FROM {my_db}.RAW.SUPPLIER_PRODUCTS_SHAPED', 1),
    ('SUPPLIER_PRODUCTS_PARTIAL loaded', f'SELECT COUNT(*) FROM {my_db}.RAW.SUPPLIER_PRODUCTS_PARTIAL', 1),
    ('LOYALTY_EVENTS_RAW loaded', f'SELECT COUNT(*) FROM {my_db}.RAW.LOYALTY_EVENTS_RAW', 1),
]
all_passed = True
for label, sql, min_val in checks:
    try:
        result = session.sql(sql).collect()[0][0]
        ok = result >= min_val
        print(('✅' if ok else '⛔') + f' {label}: {result}')
        if not ok:
            all_passed = False
    except Exception as e:
        print(f'⛔ {label}: ERROR — {e}')
        all_passed = False

# Task check: SHOW TASKS returns one row per matching task (real check, not an auto-pass)
try:
    task_rows = session.sql(f"SHOW TASKS LIKE 'LOAD_SIGNUPS_TASK' IN SCHEMA {my_db}.RAW").collect()
    ok = len(task_rows) >= 1
    print(('✅' if ok else '⛔') + f' LOAD_SIGNUPS_TASK exists: {len(task_rows)}')
    if not ok:
        all_passed = False
except Exception as e:
    print(f'⛔ LOAD_SIGNUPS_TASK: ERROR — {e}')
    all_passed = False

if all_passed:
    print('\n🎉 All checks passed.')


---
### Check your understanding 🧠

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_engineer_quiz as quiz
quiz.display_question("q3")

---
### :snowflake: Key Takeaways

- If you don't define a file format, Snowflake assumes **CSV** (comma-delimited fields, newline-delimited rows) — spell out `TYPE` and `SKIP_HEADER` when your file differs
- `VALIDATION_MODE=RETURN_ALL_ERRORS` is a dry-run — finds all bad rows before anything is written
- `ON_ERROR=CONTINUE` loads clean rows and skips bad ones — use `VALIDATE(..., JOB_ID => '_last')` to recover the rejected rows and `COPY_HISTORY` to audit the load
- VARIANT columns store semi-structured data natively — no schema required at write time
- Use **colon-dot path** (`v:field.nested_field::TYPE`) to extract and cast JSON values relationally
- A **task** schedules any single SQL statement on an interval (or CRON); new tasks start **suspended**, are resumed with `ALTER TASK ... RESUME`, and can be fired on demand with `EXECUTE TASK`
- Internal stages hold files inside Snowflake; **external stages** point at S3/GCS/Azure — same COPY INTO syntax

**The Other Personas:**
- The **Data Analyst** notebook covers using SIGNUPS_BY_CHANNEL and joining to the product catalog
- The **Platform Admin** notebook covers SHOW STAGES, COPY_HISTORY, grants, and the storage integration admin walkthrough

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.